# SpikeLM spiking text encoder — loading weights

The text half of the grounding model. This notebook builds `SpikeLMTextEncoder`,
initialises it from a pretrained checkpoint, and verifies the result actually carries
language knowledge.

## The weights situation

**SpikeLM ships no weights.** The fork contains no `.bin`/`.pt`/`.safetensors` files
anywhere, its `base_spike/` directory holds only `args.json`, its `bert-base-uncased/`
directory is empty, and the README gives no download link — the paper expects you to
pretrain from scratch on Wikipedia + BookCorpus.

Left random, the text encoder would start with no linguistic knowledge at all and would
have to learn English from the grounding objective alone. Since SpikeLM is
BERT-*architecture* with standard parameter names, a real ANN checkpoint transfers almost
completely instead: **195 tensors match by name and shape**, covering the whole 12-layer
transformer plus word embeddings, and 2 more convert cleanly.

The donor is **roberta-base**, and that choice is forced rather than arbitrary:
`Talk2EventDataset` builds `positive_map` against roberta-base tokens, so the encoder must
share that vocabulary or the alignment supervision points at the wrong words. Word
embeddings are only meaningful for the vocabulary they were trained on.

> **CUDA-only.** `SpikeLinear.forward` hard-codes `.cuda()` in the frozen fork, so the
> encoder cannot run on CPU. Weight *loading* works on CPU; only the forward pass needs a GPU.

In [1]:
import torch
import torch.nn.functional as F

from spiketrandvg.models.text_encoder import (
    MAX_TEXT_LEN,
    SPIKELM_T,
    SpikeLMTextEncoder,
    build_tokenizer,
    load_pretrained_weights,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
D_MODEL = 256
LAYERS = 12
DONOR = "roberta-base"

torch.manual_seed(0)
print(f"device {DEVICE} | d_model {D_MODEL} | layers {LAYERS} | "
      f"SpikeLM internal T {SPIKELM_T} | max text len {MAX_TEXT_LEN}")

/home/karthik/Desktop/PhD/talk2events_research/spiketrandvg/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device cuda | d_model 256 | layers 12 | SpikeLM internal T 4 | max text len 80


## 1. Confirm there is nothing to load from the fork

Before reaching for a substitute, check the claim directly.

In [2]:
from spiketrandvg.utils import forks

spikelm_root = forks.FORKS["spikelm"]
weights = [p for pat in ("*.bin", "*.pt", "*.pth", "*.safetensors", "*.ckpt")
           for p in spikelm_root.rglob(pat) if ".git" not in p.parts]
print(f"fork: {spikelm_root}")
print(f"weight files found: {len(weights)}")
for d in ("base_spike", "bert-base-uncased"):
    sub = spikelm_root / "spikeLM-BERT" / d
    if sub.is_dir():
        print(f"  {d}/: {[p.name for p in sub.iterdir()] or '(empty)'}")

fork: /home/karthik/Desktop/PhD/talk2events_research/repositories/SpikeLM
weight files found: 0
  base_spike/: ['args.json']
  bert-base-uncased/: (empty)


## 2. Build the encoder and load pretrained weights

`load_pretrained_weights` handles the two tensors that do not match shape:

- **position embeddings** `(514, 768) -> (512, 768)`. RoBERTa reserves ids 0 and 1 for its
  padding offset, so rows 2 onward are the real positions and that is the slice taken.
- **token type embeddings** `(1, 768) -> (2, 768)`. RoBERTa has a single segment; it is
  replicated across BERT's two.

SpikeLM's own spiking parameters — the per-timestep `act_clip_val` / `clip_key` /
`clip_value` lists and `weight_clip_val` buffers — have no counterpart and keep their
initialisation. That is correct: they are calibrated lazily from the statistics of the
first forward pass.

In [3]:
enc = SpikeLMTextEncoder(d_model=D_MODEL, num_hidden_layers=LAYERS)

emb_before = enc.encoder.embeddings.word_embeddings.weight.detach().clone()
report = load_pretrained_weights(enc, donor=DONOR)
emb_after = enc.encoder.embeddings.word_embeddings.weight.detach()

print()
for k, v in report.items():
    print(f"  {k}: {v}")
print(f"\nword embeddings changed: {not torch.equal(emb_before, emb_after)}")
print(f"params: {sum(p.numel() for p in enc.parameters())/1e6:.1f}M")

enc = enc.to(DEVICE)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[text_encoder] initialised from 'roberta-base': loaded 197 tensors (2 converted), 457 SpikeLM-specific params kept at init
    embeddings.position_embeddings.weight: (514, 768) -> (512, 768) (dropped ids 0,1)
    embeddings.token_type_embeddings.weight: (1, 768) -> (2, 768) (segment replicated)

  donor_tensors: 199
  loaded: 197
  converted: 2
  spiking_params_left_at_init: 457

word embeddings changed: True
params: 124.3M


## 3. Did the language knowledge actually arrive?

A load that reports success but transfers noise is the failure mode worth catching. The
test: nearest neighbours in embedding space should be semantically related, and they
should be words relevant to driving scenes.

In [4]:
tok = build_tokenizer()
W = enc.encoder.embeddings.word_embeddings.weight.detach()


def neighbours(word, k=6):
    ids = tok(word, add_special_tokens=False)["input_ids"]
    sim = F.cosine_similarity(W[ids[0]:ids[0]+1], W, dim=1)
    return [tok.decode([j]).strip() for j in sim.topk(k + 1).indices[1:]]


for w in [" car", " pedestrian", " truck", " left", " moving", " road"]:
    print(f"{w.strip():12} -> {', '.join(neighbours(w))}")

car          -> cars, vehicle, Car, Car, car, automobile
pedestrian   -> pedestrians, sidewalk, commuter, sidewalks, passenger, roadway
truck        -> trucks, Truck, vehicle, car, vehicles, SUV
left         -> Left, left, Left, leave, leaving, leaves
moving       -> Moving, moved, move, Moving, moves, moving
road         -> Road, roads, Road, roadway, highway, road


## 4. Real Talk2Event captions

Encode actual referring expressions and check the shapes line up with what the fusion
stage and the `positive_map` supervision expect.

In [5]:
from types import SimpleNamespace

from spiketrandvg.datasets.talk2event_dataset import Talk2EventDataset

ds = Talk2EventDataset(SimpleNamespace(attribute="all"), image_set="test")

# the dataset's tokenizer and ours must agree exactly, or positive_map misaligns
agree = sum(ds.tokenizer(ds[i][1]["caption"])["input_ids"]
            == tok(ds[i][1]["caption"])["input_ids"] for i in range(100))
print(f"tokenizer agreement on 100 real captions: {agree}/100")

caps = [ds[i][1]["caption"] for i in range(4)]
batch = tok(caps, padding="max_length", truncation=True,
            max_length=MAX_TEXT_LEN, return_tensors="pt").to(DEVICE)

enc.eval()
with torch.no_grad():
    tokens, sentence = enc(batch["input_ids"], batch["attention_mask"])

print(f"\ntokens   {tuple(tokens.shape)}  (B, L, d_model) -> pairs with positive_map")
print(f"sentence {tuple(sentence.shape)}  (B, d_model) -> conditions the vision branch")
print(f"real token counts per caption: {batch['attention_mask'].sum(1).tolist()}")

Initializing Talk2EventDataset
missing 0 attributes
load 7665 data from test split
false match 0 attributes
tokenizer agreement on 100 real captions: 100/100

tokens   (4, 80, 256)  (B, L, d_model) -> pairs with positive_map
sentence (4, 256)  (B, d_model) -> conditions the vision branch
real token counts per caption: [40, 39, 38, 39]


## 5. Does pretraining buy anything measurable?

The point of loading weights is discrimination: differently-worded expressions should
produce different representations. Comparing a random-init encoder against the
pretrained one on the same captions makes the benefit concrete rather than assumed.

Captions 0–2 describe the **same object** in three phrasings; caption 3 is a different
sample. A useful encoder should rate the first three as more similar to each other than
to the fourth.

In [6]:
def sentence_vectors(model):
    model.eval()
    with torch.no_grad():
        return model(batch["input_ids"], batch["attention_mask"])[1]


rand_enc = SpikeLMTextEncoder(d_model=D_MODEL, num_hidden_layers=LAYERS).to(DEVICE)
s_rand = sentence_vectors(rand_enc)
s_pre = sentence_vectors(enc)


def pair_table(s, name):
    print(f"\n{name}: pairwise cosine between the 4 captions")
    n = s.shape[0]
    for a in range(n):
        row = [f"{F.cosine_similarity(s[a:a+1], s[b:b+1], dim=1).item():+.3f}" for b in range(n)]
        print("   " + "  ".join(row))
    same = [F.cosine_similarity(s[a:a+1], s[b:b+1], dim=1).item()
            for a in range(3) for b in range(a + 1, 3)]
    diff = [F.cosine_similarity(s[a:a+1], s[3:4], dim=1).item() for a in range(3)]
    print(f"   mean within same object : {sum(same)/len(same):+.3f}")
    print(f"   mean vs different object: {sum(diff)/len(diff):+.3f}")
    print(f"   separation (higher is better): {sum(same)/len(same) - sum(diff)/len(diff):+.3f}")


pair_table(s_rand, "RANDOM init")
pair_table(s_pre, "PRETRAINED (roberta-base)")

del rand_enc
if torch.cuda.is_available():
    torch.cuda.empty_cache()


RANDOM init: pairwise cosine between the 4 captions
   +1.000  +0.940  +0.930  +0.942
   +0.940  +1.000  +0.923  +0.929
   +0.930  +0.923  +1.000  +0.933
   +0.942  +0.929  +0.933  +1.000
   mean within same object : +0.931
   mean vs different object: +0.935
   separation (higher is better): -0.004

PRETRAINED (roberta-base): pairwise cosine between the 4 captions
   +1.000  +0.960  +0.954  +0.948
   +0.960  +1.000  +0.942  +0.957
   +0.954  +0.942  +1.000  +0.947
   +0.948  +0.957  +0.947  +1.000
   mean within same object : +0.952
   mean vs different object: +0.951
   separation (higher is better): +0.001


### 5b. Is the low separation a *spiking* problem?

The numbers above are easy to misread as "the spiking quantisation destroyed the pretrained
knowledge". Before believing that, measure the **ceiling**: what does the plain ANN
roberta-base achieve on the same captions with the same mean-pooling?

If the ANN reference scores no better, the flat similarities are a property of the data and
the pooling — Talk2Event captions are highly homogeneous, all describing road users with
spatial relations in near-identical vocabulary — and not evidence against the spiking path.

In [7]:
from transformers import AutoModel

# 6 captions: 0-2 describe one object, 3-5 describe another
idx = [0, 1, 2, 3, 4, 5]
caps6 = [ds[i][1]["caption"] for i in idx]
b6 = tok(caps6, padding="max_length", truncation=True,
         max_length=MAX_TEXT_LEN, return_tensors="pt").to(DEVICE)
m6 = b6["attention_mask"].unsqueeze(-1).float()


def separation(vecs, label):
    v = F.normalize(vecs, dim=-1)
    within = [(v[a] @ v[b]).item()
              for a, b in [(0, 1), (0, 2), (1, 2), (3, 4), (3, 5), (4, 5)]]
    across = [(v[a] @ v[b]).item() for a in range(3) for b in range(3, 6)]
    w, x = sum(within) / len(within), sum(across) / len(across)
    print(f"  {label:34} within {w:+.3f}  across {x:+.3f}  separation {w - x:+.3f}")
    return w - x


ann = AutoModel.from_pretrained("roberta-base").to(DEVICE).eval()
with torch.no_grad():
    h = ann(input_ids=b6["input_ids"], attention_mask=b6["attention_mask"]).last_hidden_state
ceiling = separation((h * m6).sum(1) / m6.sum(1), "ANN roberta-base (ceiling)")

with torch.no_grad():
    hs = enc.encoder(input_ids=b6["input_ids"],
                     attention_mask=b6["attention_mask"]).last_hidden_state
separation((hs * m6).sum(1) / m6.sum(1), "SpikeLM hidden (before proj)")
with torch.no_grad():
    spiking = separation(enc(b6["input_ids"], b6["attention_mask"])[1],
                         "SpikeLM after random proj")

print(f"\nThe spiking encoder tracks the ANN ceiling ({spiking:+.3f} vs {ceiling:+.3f}).")
print("So the flat similarities are about homogeneous captions and mean-pooling,")
print("NOT about the spiking quantisation losing the pretrained weights.")

del ann
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  ANN roberta-base (ceiling)         within +0.995  across +0.993  separation +0.002
  SpikeLM hidden (before proj)       within +0.959  across +0.961  separation -0.002
  SpikeLM after random proj          within +0.955  across +0.958  separation -0.003

The spiking encoder tracks the ANN ceiling (-0.003 vs +0.002).
So the flat similarities are about homogeneous captions and mean-pooling,
NOT about the spiking quantisation losing the pretrained weights.


## Notes

- The projection `proj` (hidden 768 -> `d_model` 256) is **always** randomly initialised —
  it has no donor counterpart by construction, since it exists to match the fusion width.
- `enc.config.T` is SpikeLM's own spiking time axis (4 by default) and is *independent* of
  the vision encoder's `T_STEPS = 5`. `BertEncoder` repeats internally and averages over T
  before returning, so a caption yields one static representation reused across every
  vision timestep. That is the right semantics: the expression does not change over the
  event sequence.
- Pass `freeze=True` to `SpikeLMTextEncoder` to hold the language encoder fixed and train
  only `proj` — worth trying if the grounding task starts overfitting the text side.
- To save an initialised encoder for reuse:
  `torch.save(enc.state_dict(), "ckpts/spikelm_text_roberta_init.pth")`.